# Gemini API + Google Search Grounding 테스트

In [2]:
import sys
sys.path.insert(0, '..')

from dotenv import load_dotenv
load_dotenv('../config/api_keys.env')

import os
API_KEY = os.getenv('GEMINI_API_KEY')
print('API Key loaded:', API_KEY[:10] + '...' if API_KEY else 'NOT FOUND')

API Key loaded: AIzaSyDTEH...


## 1. 기본 호출 테스트 (검색 없음)

In [3]:
from google import genai
from google.genai import types

client = genai.Client(api_key=API_KEY)
MODEL = 'gemini-3.1-pro-preview'

resp = client.models.generate_content(
    model=MODEL,
    contents='안녕? 간단히 한 줄로 답해줘.',
)
print('모델:', MODEL)
print('응답:', resp.text)

모델: gemini-3.1-pro-preview
응답: 안녕하세요! 무엇을 도와드릴까요?


## 2. Google Search Grounding 테스트

In [4]:
resp = client.models.generate_content(
    model=MODEL,
    contents='오늘 미국 주식시장에 영향을 줄 만한 주요 뉴스 3가지를 한국어로 요약해줘.',
    config=types.GenerateContentConfig(
        tools=[types.Tool(google_search=types.GoogleSearch())],
        temperature=0.3,
    ),
)

print('=== 응답 ===')  
print(resp.text)

# 검색 쿼리 수 확인
try:
    meta = resp.candidates[0].grounding_metadata
    queries = meta.web_search_queries if meta else []
    print(f'\n=== 사용된 검색 쿼리 ({len(queries)}개) ===')
    for q in queries:
        print(f'  - {q}')
except Exception as e:
    print(f'메타데이터 없음: {e}')

=== 응답 ===
2026년 3월 27일 현재, 미국 주식시장에 큰 영향을 미치고 있는 주요 뉴스 3가지는 다음과 같습니다.

**1. 미국-이란 전쟁 장기화 우려 및 국제 유가 급등으로 인한 증시 급락**
도널드 트럼프 미국 대통령이 이란 에너지 시설에 대한 타격 유예 기간을 4월 6일까지 열흘 연장했음에도 불구하고, 양국 간의 휴전 협상 타결에 대한 기대감이 꺾이면서 시장의 불안이 커지고 있습니다. 이로 인해 브렌트유가 배럴당 100달러를 다시 돌파하고 서부텍사스산원유(WTI)가 94달러를 넘어서는 등 국제 유가가 급등했습니다. 유가상승이 인플레이션을 다시 자극할 것이라는 우려 속에 다우지수(-1.01%), S&P 500(-1.74%), 나스닥(-2.38%) 등 3대 지수가 일제히 급락했으며, 특히 기술주 중심의 나스닥 지수는 전고점 대비 10% 이상 하락하며 조정 국면에 진입했습니다.

**2. '소셜미디어 중독' 소송 패소 여파로 메타·알파벳 등 빅테크 주가 하락**
인스타그램(메타)과 유튜브(알파벳)가 청소년의 소셜미디어 중독 등 정신 건강 피해에 책임이 있다는 미국 배심원단의 첫 랜드마크 평결이 나왔습니다. 부과된 벌금 규모 자체는 기업 이익 대비 크지 않으나, 향후 수천 건의 유사한 줄소송으로 이어질 수 있다는 우려가 투자 심리를 크게 위축시켰습니다. 이 여파로 메타 주가가 8% 급락하고 알파벳이 3.4% 하락했으며, 엔비디아(-4.2%), 아마존(-2%) 등 주요 대형 기술주들이 동반 하락하며 증시 전체를 끌어내리는 주요 원인이 되었습니다.

**3. 견조한 고용 지표와 인플레이션 우려로 연준(Fed)의 금리 인하 기대감 후퇴**
미국 노동부가 발표한 주간 신규 실업수당 청구 건수가 21만 건으로 시장 예상치에 부합하며, 2년 만에 최저 수준의 안정적인 노동 시장 흐름을 보여주었습니다. 고용이 탄탄하게 버텨주는 가운데 중동 전쟁으로 인한 유가 충격이 물가 상승 압력을 높이고 있어, 시장에서는 연방준비제도(Fed)가 당분간 기준금리를 인하하기보다

## 3. Flash-Lite Fallback 테스트

In [5]:
FALLBACK = 'gemini-3.1-flash-lite-preview'

resp = client.models.generate_content(
    model=FALLBACK,
    contents='오늘 미국 증시 관련 뉴스 2가지 요약해줘.',
    config=types.GenerateContentConfig(
        tools=[types.Tool(google_search=types.GoogleSearch())],
        temperature=0.3,
    ),
)
print('모델:', FALLBACK)
print(resp.text)

try:
    queries = resp.candidates[0].grounding_metadata.web_search_queries or []
    print(f'검색 쿼리: {queries}')
except:
    pass

모델: gemini-3.1-flash-lite-preview
2026년 3월 27일 기준, 미국 증시와 관련된 주요 뉴스 2가지를 요약해 드립니다.

### 1. 중동 지정학적 리스크로 인한 증시 하락 및 나스닥 조정
미국과 이란 간의 갈등이 지속되면서 투자 심리가 크게 위축되었습니다. 지난 26일(현지시간) 뉴욕 증시에서 다우존스, S&P 500, 나스닥 등 3대 지수가 일제히 하락했습니다. 특히 나스닥 지수는 고점 대비 10% 이상 하락하며 기술적 '조정' 국면에 진입했습니다. 시장은 양국 간의 협상 불확실성과 전쟁 장기화에 따른 인플레이션 우려를 경계하고 있습니다.

### 2. 국제 유가 급등과 트럼프 대통령의 이란 관련 발언
지정학적 긴장감으로 인해 국제 유가가 큰 폭으로 상승하며 시장에 부담을 주었습니다. 도널드 트럼프 대통령은 이란에 대한 에너지 시설 공격 보류 시한을 10일 연장한다고 밝혔으나, 동시에 이란의 석유 통제권 문제 등을 언급하며 강경한 입장을 유지했습니다. 이러한 불확실성 속에서 유가 변동성이 커졌고, 이는 인플레이션 우려를 자극하며 주식 시장의 하락 압력으로 작용했습니다.

---
*참고: 위 내용은 2026년 3월 27일 시점의 시장 상황을 바탕으로 요약되었습니다.*
검색 쿼리: ['US stock market news March 27 2026', '오늘 미국 증시 주요 뉴스 2026년 3월 27일']


## 4. JSON 모드 + Search 동시 가능한지 테스트

In [6]:
import json

prompt = '''오늘 미국 주식시장에 영향을 줄 주요 매크로 이벤트를 검색하고,
아래 JSON 형식으로 2개 반환해줘:
[{"event": "이벤트명", "impact": "bullish|bearish", "sectors": ["섹터1"]}]'''

try:
    resp = client.models.generate_content(
        model=MODEL,
        contents=prompt,
        config=types.GenerateContentConfig(
            tools=[types.Tool(google_search=types.GoogleSearch())],
            response_mime_type='application/json',
            temperature=0.2,
        ),
    )
    print('JSON + Search 동시 지원: ✅')
    data = json.loads(resp.text)
    print(json.dumps(data, ensure_ascii=False, indent=2))
except Exception as e:
    print(f'JSON + Search 동시 지원: ❌')
    print(f'에러: {e}')
    print('→ Search만 하고 JSON은 파싱으로 처리해야 함')

JSON + Search 동시 지원: ✅
[
  {
    "event": "미시간대 소비자심리지수 (Michigan Consumer Sentiment)",
    "impact": "bullish",
    "sectors": [
      "Consumer Discretionary",
      "Retail"
    ]
  },
  {
    "event": "미시간대 기대인플레이션 (U.S. Inflation Expectations)",
    "impact": "bearish",
    "sectors": [
      "Technology",
      "Real Estate"
    ]
  }
]
